# Milestone 2: Full Notebook (Q1 to Q8)
**Part 1:** Q1 to Q6 (RLE and EDA). **Part 2:** Q7 and Q8 (Dataset, U-Net from scratch, training).
Attach the competition data and turn on a **GPU** before running. Run cells top to bottom.

## 0. Setup and RLE helpers

In [1]:
import os, glob, itertools, random, time
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# PATHS: auto-detected on Kaggle. If detection fails, set them by hand:
# CSV_PATH = "/kaggle/input/<competition-slug>/train.csv"
# IMG_DIR  = "/kaggle/input/<competition-slug>/train_images"
# ------------------------------------------------------------------
CSV_PATH = None
IMG_DIR = None

if CSV_PATH is None:
    hits = sorted(glob.glob("/kaggle/input/**/train*.csv", recursive=True))
    print("train csv candidates:", hits)
    CSV_PATH = hits[0] if hits else "train.csv"
print("Using CSV:", CSV_PATH)

H, W = 128, 800
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]

def has_mask(rle):
    """True if an RLE cell holds a real mask. Empty cells can be NaN, '' or ' '."""
    return isinstance(rle, str) and rle.strip() != ""

def rle_decode(rle, h=H, w=W):
    """RLE string -> binary (h, w) mask. Indexing is 1-based, traversal is column-major."""
    flat = np.zeros(h * w, dtype=np.uint8)
    if has_mask(rle):
        nums = np.array(rle.split(), dtype=np.int64)
        starts = nums[0::2] - 1          # 1-indexed -> 0-indexed
        lengths = nums[1::2]
        for s, l in zip(starts, lengths):
            flat[s:s + l] = 1
    return flat.reshape((h, w), order="F")   # F = column-major

def rle_encode(mask):
    """Binary mask -> RLE string (1-indexed, column-major). Needed for the Kaggle submission later."""
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(map(str, runs))

df = pd.read_csv(CSV_PATH)
print(df.shape, "| unique images:", df.image_id.nunique())
df.head()

train csv candidates: ['/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv']
Using CSV: /kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv
(45572, 5) | unique images: 11393


,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128


In [2]:
print(df[["width", "height"]].drop_duplicates())   # expect a single row: 800 x 128

   width  height
0    800     128


## Q1: RLE of the 8 x 8 grid
In 1-based column-major order the printed cell number *is* the flat index, so we only need the runs of consecutive numbers.

**Verify the `cells` list against the original figure before submitting.**

In [3]:
cells = [12, 13, 23, 24, 25, 26, 44, 45, 47]   # VERIFY against the original figure!
flat = np.zeros(64, dtype=np.uint8)
flat[np.array(cells) - 1] = 1
grid = flat.reshape((8, 8), order="F")
print(grid)

q1 = rle_encode(grid)
print("Q1 answer:", q1)          # expected: 12 2 23 4 44 2 47 1
assert (rle_decode(q1, 8, 8) == grid).all(), "round trip failed"

[[0 0 0 1 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0]
 [0 1 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0]
 [0 0 1 0 0 0 0 0]]
Q1 answer: 12 2 23 4 44 2 47 1


## Q2: Decode all 4 masks of image `0ZUU8LH57Y` and sum the pixels

In [4]:
rows = df[df.image_id == "0ZUU8LH57Y"]
print(rows[["defect_type", "mask_rle"]])

total = 0
for d in DEFECTS:
    r = rows.loc[rows.defect_type == d, "mask_rle"]
    rle = r.iloc[0] if len(r) else None
    m = rle_decode(rle)
    print(d, m.shape, int(m.sum()))
    total += int(m.sum())
print("Q2 answer:", total)

      defect_type                                           mask_rle
40804    defect_a  88169 11 88183 6 88294 24 88422 24 88550 24 88...
40805    defect_b                                                NaN
40806    defect_c  37642 1 37769 2 37897 3 38025 4 38153 5 38281 ...
40807    defect_d  79113 13 79239 16 79367 17 79495 18 79622 21 7...
defect_a (128, 800) 250
defect_b (128, 800) 0
defect_c (128, 800) 1232
defect_d (128, 800) 5032
Q2 answer: 6514


## Q3 and Q4: Class prevalence and co-occurrence

In [5]:
# one row per image, one column per defect
wide = df.pivot(index="image_id", columns="defect_type", values="mask_rle")[DEFECTS]
P = wide.map(has_mask).astype(int)          # presence matrix (n_images, 4)
print("presence matrix:", P.shape)

# Q3
counts = P.sum()
print(counts)
print(f"Q3 answer: {counts.idxmax()} ({counts.max()}), {counts.idxmin()} ({counts.min()})")

# Q4: C = P^T P  ->  C_ij = #images with both i and j, diagonal = per-class image count
C = pd.DataFrame(P.values.T @ P.values, index=DEFECTS, columns=DEFECTS)
print(C)

best = max(itertools.combinations(DEFECTS, 2), key=lambda p: C.loc[p[0], p[1]])
print(f"Q4 answer: {best[0]}-{best[1]}, {C.loc[best[0], best[1]]}")

presence matrix: (11393, 4)
defect_type
defect_a    1081
defect_b     313
defect_c    4100
defect_d    1707
dtype: int64
Q3 answer: defect_c (4100), defect_b (313)
          defect_a  defect_b  defect_c  defect_d
defect_a      1081        42       310       234
defect_b        42       313        95        64
defect_c       310        95      4100       705
defect_d       234        64       705      1707
Q4 answer: defect_c-defect_d, 705


## Q5: Co-occurrence vs correlation (base-rate effect)
Pearson correlation between the binary presence columns (the phi coefficient). It is high only if two defects appear together *more often than their individual frequencies would predict*.

In [6]:
corr = pd.DataFrame(np.corrcoef(P.values.T), index=DEFECTS, columns=DEFECTS)
print(corr.round(2))

i, j = best
r = corr.loc[i, j]
print(f"Q5 correlation for {i}-{j}: {r:.2f}")
print("Strong association sustained?", "Yes" if abs(r) >= 0.5 else "No")

          defect_a  defect_b  defect_c  defect_d
defect_a      1.00      0.02     -0.05      0.06
defect_b      0.02      1.00     -0.02      0.03
defect_c     -0.05     -0.02      1.00      0.05
defect_d      0.06      0.03      0.05      1.00
Q5 correlation for defect_c-defect_d: 0.05
Strong association sustained? No


## Q6: Pixel-level overlap between defect classes
Decode masks of every multi-label image on the native 128 x 800 canvas and count overlapping pixels for each pair.

In [7]:
multi_ids = P.index[P.sum(axis=1) >= 2]
print("multi-label images:", len(multi_ids))

max_overlap, max_dice, argmax = 0, 0.0, None
for iid in multi_ids:
    present = [d for d in DEFECTS if P.loc[iid, d]]
    masks = {d: rle_decode(wide.loc[iid, d]) for d in present}
    for a, b in itertools.combinations(present, 2):
        inter = int((masks[a] & masks[b]).sum())
        denom = int(masks[a].sum() + masks[b].sum())
        dice = 2 * inter / denom if denom else 0.0
        if inter > max_overlap:
            max_overlap, argmax = inter, (iid, a, b)
        max_dice = max(max_dice, dice)

print("max overlap pixels:", max_overlap, "| max dice:", round(max_dice, 4), "| at:", argmax)
print("Q6 answer:", max_overlap, "Yes" if max_overlap == 0 else "No")   # Yes = mutually exclusive

multi-label images: 1198
max overlap pixels: 0 | max dice: 0.0 | at: None
Q6 answer: 0 Yes


---
# PART 2: Q7 and Q8 (Dataset, U-Net from scratch, training)

## Part 2 setup (extra imports and config)

In [8]:
# ---- extra imports for Part 2 ----
import cv2
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

SEED, EPOCHS, LR, BATCH_SIZE = 42, 5, 3e-4, 16
USE_WANDB = False      # set True (and wandb.login()) when you want experiment tracking
WANDB_PROJECT = "YourRollNo-t32026"

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [9]:
# Locate the image folder: the directory whose file names match the image_ids in the csv
if IMG_DIR is None:
    ids = set(df.image_id.unique())
    for root, dirs, files in os.walk("/kaggle/input"):
        imgs = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
        if imgs and Path(imgs[0]).stem in ids:
            IMG_DIR = root
            break
print("Using IMG_DIR:", IMG_DIR)
IMG_PATHS = {p.stem: p for p in Path(IMG_DIR).iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")}
print("images found:", len(IMG_PATHS))

Using IMG_DIR: /kaggle/input/competitions/industrial-surface-defect-detection/public/train
images found: 11393


## Q7: Dataset for Paradigm 1 (multi-channel joint prediction)
Input is a `(3, 128, 800)` float tensor in [0, 1]. Target is a `(4, 128, 800)` float tensor, channels 0..3 = defect_a..d.

In [10]:
class SurfaceAnomalyDataset(Dataset):
    def __init__(self, df, img_paths=IMG_PATHS, image_ids=None):
        self.img_paths = img_paths
        self.image_ids = sorted(df.image_id.unique()) if image_ids is None else list(image_ids)
        sub = df[df.image_id.isin(self.image_ids)]
        self.rles = {(i, d): r for i, d, r in zip(sub.image_id, sub.defect_type, sub.mask_rle)}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        iid = self.image_ids[idx]
        img = cv2.imread(str(self.img_paths[iid]))            # OpenCV loads BGR
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # -> RGB
        img = img.astype(np.float32) / 255.0                  # [0, 1]
        img = torch.from_numpy(img).permute(2, 0, 1)          # HWC -> CHW = (3, 128, 800)

        masks = np.stack([rle_decode(self.rles.get((iid, d))) for d in DEFECTS])  # (4, 128, 800)
        masks = torch.from_numpy(masks).to(torch.float32)
        return img, masks

full_ds = SurfaceAnomalyDataset(df)
loader = DataLoader(full_ds, batch_size=4, shuffle=False)
images, masks = next(iter(loader))
print("images:", tuple(images.shape), images.dtype)   # (4, 3, 128, 800) torch.float32
print("masks :", tuple(masks.shape), masks.dtype)     # (4, 4, 128, 800) torch.float32

images: (4, 3, 128, 800) torch.float32
masks : (4, 4, 128, 800) torch.float32


**Q7 (2): why float32?** `nn.BCEWithLogitsLoss` computes `-[y*log(sigmoid(x)) + (1-y)*log(1-sigmoid(x))]`. The logits `x` are float32 and the target `y` must have the same floating dtype for that arithmetic, so uint8/bool targets raise a dtype error. Float targets also allow soft labels.

**To try (Paradigm 2):** `__len__` becomes `n_images x 4` (or balanced sampling of `(image, class)` pairs); `__getitem__` returns `(image, class_id, mask)` with mask shaped `(1, 128, 800)`; the model needs a class-embedding input.

## Stratified train / validation split
Stratify on the *pattern* of defects (e.g. `0101`) so rare combinations appear in both sets.

In [11]:
wide = df.pivot(index="image_id", columns="defect_type", values="mask_rle")[DEFECTS]
pattern = wide.map(has_mask).astype(int).astype(str).agg("".join, axis=1)
vc = pattern.value_counts()
pattern = pattern.where(pattern.map(vc) >= 10, "rare")   # merge tiny groups so stratify works

train_ids, val_ids = train_test_split(wide.index, test_size=0.2, random_state=SEED, stratify=pattern)

train_ds = SurfaceAnomalyDataset(df, image_ids=train_ids)
val_ds   = SurfaceAnomalyDataset(df, image_ids=val_ids)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("train:", len(train_ds), "| val:", len(val_ds))

train: 9114 | val: 2279


## Q8: 3-stage U-Net built from scratch

In [12]:
class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2. padding=1 keeps height and width unchanged."""
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class UNet3(nn.Module):
    """3 down-sampling stages + bottleneck + 3 up-sampling stages with skip connections."""
    def __init__(self, in_ch=3, out_ch=4, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)            # 128 x 800
        self.enc2 = DoubleConv(base, base * 2)         # 64 x 400
        self.enc3 = DoubleConv(base * 2, base * 4)     # 32 x 200
        self.bott = DoubleConv(base * 4, base * 8)     # 16 x 100
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)         # 4 logits per pixel (no sigmoid here)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bott(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)

model = UNet3().to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print("output shape:", tuple(model(torch.randn(2, 3, H, W, device=device)).shape))   # (2, 4, 128, 800)

parameters: 1,927,108
output shape: (2, 4, 128, 800)


## Training and validation (5 epochs, AdamW, lr 3e-4, BCEWithLogits, mixed precision)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name="unet3-scratch-bce",
               config=dict(epochs=EPOCHS, lr=LR, batch=BATCH_SIZE, base=32, loss="BCEWithLogits"))

@torch.no_grad()
def batch_stats(logits, masks, thr=0.5):
    """Proxy for the competition metric (exact FP / empty formulas are not public)."""
    pred = (torch.sigmoid(logits) > thr).float()
    inter = (pred * masks).sum((2, 3)); ps = pred.sum((2, 3)); gs = masks.sum((2, 3))
    pos = gs > 0
    dice = 2 * inter / (ps + gs + 1e-9)
    return dict(dice_sum=dice[pos].sum().item(), n_pos=pos.sum().item(),
                fp_sum=(ps[~pos] / (H * W)).sum().item(),
                empty_ok=(ps[~pos] == 0).sum().item(), n_neg=(~pos).sum().item())

def run_epoch(loader, train):
    model.train(train)
    tot_loss, n = 0.0, 0
    st = dict(dice_sum=0, n_pos=0, fp_sum=0, empty_ok=0, n_neg=0)
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.float(), y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            for k, v in batch_stats(logits.float(), y).items():
                st[k] += v
        tot_loss += loss.item() * x.size(0)
        n += x.size(0)
    out = dict(loss=tot_loss / n)
    if not train:
        dice = st["dice_sum"] / max(st["n_pos"], 1)
        fp = 1 - st["fp_sum"] / max(st["n_neg"], 1)
        emp = st["empty_ok"] / max(st["n_neg"], 1)
        out.update(dice=dice, score=0.7 * dice + 0.2 * fp + 0.1 * emp)
    return out

history, best_val = [], float("inf")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    history.append(dict(epoch=epoch, train_loss=tr["loss"], val_loss=va["loss"],
                        val_dice=va["dice"], val_score=va["score"]))
    if va["loss"] < best_val:
        best_val = va["loss"]
        torch.save(model.state_dict(), "best_unet3.pt")
    print(f"Epoch {epoch}/{EPOCHS} | train {tr['loss']:.4f} | val {va['loss']:.4f} | "
          f"dice {va['dice']:.3f} | proxy score {va['score']:.3f} | {time.time()-t0:.0f}s")
    if USE_WANDB:
        wandb.log(history[-1])

hist = pd.DataFrame(history)
hist.to_csv("history.csv", index=False)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## Q8 answers and loss curves

In [ ]:
e1, e5 = hist.iloc[0], hist.iloc[-1]
best_epoch = int(hist.loc[hist.val_loss.idxmin(), "epoch"])
print(f"epoch1 (train, val) = ({e1.train_loss:.4f}, {e1.val_loss:.4f})")
print(f"epoch5 (train, val) = ({e5.train_loss:.4f}, {e5.val_loss:.4f})")
print("best_epoch =", best_epoch)
print("gap at epoch 5 (val - train):", round(e5.val_loss - e5.train_loss, 4))

plt.plot(hist.epoch, hist.train_loss, marker="o", label="train")
plt.plot(hist.epoch, hist.val_loss, marker="o", label="val")
plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("loss_curves.png", dpi=120); plt.show()

**How to judge Q8 (3):**
- *Healthy generalisation:* both losses fall and val stays close to train.
- *Overfitting:* train keeps falling while val rises or flattens, and the gap widens.
- *Underfitting:* both stay high and are still falling slowly at epoch 5.

Write 1-2 sentences using your actual numbers.